In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from utils.utils import load_encrypted_xlsx

import os
os.environ["R_HOME"] = "/Library/Frameworks/R.framework/Versions/4.1/Resources"
from pymer4.models import Lmer

In [ ]:
registry_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/sos_sah_data/post_hoc_modified_aSAH_DATA_2009_2023_24122023.xlsx'
outcome_data_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/sos_sah_data/follow_up/aSAH_DATA_2009_2024_18122024.xlsx'
bp_path = "/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/pdms_data/extracted_data/20240116_SAH_SOS_Blutdruecke.csv"
registry_pdms_correspondence_path = "/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/pdms_data/registry_pdms_correspondence.csv"

In [ ]:
registry_df = load_encrypted_xlsx(registry_path)
bp_df = pd.read_csv(bp_path, sep=';', decimal='.')
registry_pdms_correspondence_df = pd.read_csv(registry_pdms_correspondence_path)

In [ ]:
outcome_df = load_encrypted_xlsx(outcome_data_path)

In [ ]:
bp_df = bp_df.merge(registry_pdms_correspondence_df, how="left", on="pNr")

In [ ]:
# <!-- fillna in mRS_FU_1y with mRS_2FU_2y and then mRS_3FU_5y -->
outcome_df['mRS_FU_1y'] = outcome_df['mRS_FU_1y'].fillna(outcome_df['mRS_2FU_2y'])
outcome_df['mRS_FU_1y'] = outcome_df['mRS_FU_1y'].fillna(outcome_df['mRS_3FU_5y'])

outcome_df['mRS_FU_1y'] = pd.to_numeric(outcome_df['mRS_FU_1y'], errors='coerce')
outcome_df['mRS_discharge'] = pd.to_numeric(outcome_df['mRS_discharge'], errors='coerce')

# if mRS_discharge == 6, set mRS_FU_1y to 6
outcome_df.loc[outcome_df['mRS_discharge'] == 6, 'mRS_FU_1y'] = 6

In [ ]:
outcome_df["mRS_FU_1y_int"] = pd.to_numeric(outcome_df["mRS_FU_1y"], errors="coerce")
bp_df["Date_birth"] = pd.to_datetime(bp_df["Date_birth"], format="%d.%m.%Y")
outcome_df["Date_birth"] = pd.to_datetime(outcome_df["Date_birth"])

In [ ]:
bp_df["mrs_1y"] = np.nan
for pnr in tqdm(bp_df["pNr"].unique()):
    sos_center_nr = bp_df[bp_df["pNr"] == pnr]["SOS-CENTER-YEAR-NO."].values[0]
    name = bp_df[bp_df["pNr"] == pnr]["JoinedName"].values[0]
    date_birth = bp_df[bp_df["pNr"] == pnr]["Date_birth"].values[0]
    mrs_values = outcome_df[(outcome_df["SOS-CENTER-YEAR-NO."] == sos_center_nr) &
                        (outcome_df["Name"] == name) &
                        (outcome_df["Date_birth"] == date_birth)]["mRS_FU_1y_int"]
    if len(mrs_values) == 0:
        mrs = np.nan
    else:
        mrs = mrs_values.values[0]

    bp_df.loc[bp_df["pNr"] == pnr, "mrs_1y"] = mrs



In [ ]:
registry_df = registry_df.drop_duplicates(subset=["SOS-CENTER-YEAR-NO.", "Date_birth", "Name"])
bp_df = bp_df.merge(registry_df[["SOS-CENTER-YEAR-NO.", "Date_birth", "Name", 'DCI_ischemia', 'Fisher_Score', 'WFNS'	]], how="left", left_on=["SOS-CENTER-YEAR-NO.", "Date_birth", "JoinedName"],
                right_on=["SOS-CENTER-YEAR-NO.", "Date_birth", "Name"])
bp_df['Fisher_Score'] = pd.to_numeric(bp_df['Fisher_Score'], errors='coerce')

In [ ]:
# get first measure (by timeBd) for every pNr
first_measure_df = bp_df.groupby("pNr").agg({"timeBd": "min"}).reset_index()
first_measure_df = first_measure_df.rename(columns={"timeBd": "first_timeBd"})
bp_df = bp_df.merge(first_measure_df, how="left", on="pNr")
bp_df["relative_timeBd"] = (pd.to_datetime(bp_df["timeBd"]) - pd.to_datetime(bp_df["first_timeBd"])).dt.total_seconds() / 3600

In [ ]:
# plot boxplot per mrs for first 24h
# 3 subplots: diastole, systole, mean
fig, ax = plt.subplots(3, 1, figsize=(10, 10))

first_24h_df = bp_df[(bp_df["relative_timeBd"] >= 0) & (bp_df["relative_timeBd"] <= 24)]

sns.boxplot(x="mrs_1y", y="systole", data=first_24h_df, hue="mrs_1y", ax=ax[0], legend=False, showfliers=False)
sns.boxplot(x="mrs_1y", y="diastole", data=first_24h_df, hue="mrs_1y", ax=ax[1], legend=False, showfliers=False)
sns.boxplot(x="mrs_1y", y="mitteldruck", data=first_24h_df, hue="mrs_1y", ax=ax[2], legend=False, showfliers=False)

plt.show()

In [ ]:
# plot mrs 0-2 and 3-6 in first 24h
# 3 subplots: diastole, systole, mean
fig, ax = plt.subplots(3, 1, figsize=(10, 10))
first_24h_df = bp_df[(bp_df["relative_timeBd"] >= 0) & (bp_df["relative_timeBd"] <= 24)]
first_24h_df["mrs_1y"] = first_24h_df["mrs_1y"].replace([0, 1, 2], "0-2")
first_24h_df["mrs_1y"] = first_24h_df["mrs_1y"].replace([3, 4, 5, 6], "3-6")
sns.boxplot(x="mrs_1y", y="systole", data=first_24h_df, hue="mrs_1y", ax=ax[0], legend=True, showfliers=False)
sns.boxplot(x="mrs_1y", y="diastole", data=first_24h_df, hue="mrs_1y", ax=ax[1], legend=False, showfliers=False)
sns.boxplot(x="mrs_1y", y="mitteldruck", data=first_24h_df, hue="mrs_1y", ax=ax[2], legend=False, showfliers=False)
plt.show()

# plot same boxplots but with DCI-ischemia as hue
fig, ax = plt.subplots(3, 1, figsize=(10, 10))
sns.boxplot(x="DCI_ischemia", y="systole", data=first_24h_df, hue="DCI_ischemia", ax=ax[0], legend=True, showfliers=False)
sns.boxplot(x="DCI_ischemia", y="diastole", data=first_24h_df, hue="DCI_ischemia", ax=ax[1], legend=False, showfliers=False)
sns.boxplot(x="DCI_ischemia", y="mitteldruck", data=first_24h_df, hue="DCI_ischemia", ax=ax[2], legend=False, showfliers=False)

In [ ]:
# - plot by fisher score and DCI

fig, ax = plt.subplots(3, 1, figsize=(10, 10))
first_24h_df = bp_df[(bp_df["relative_timeBd"] >= 0) & (bp_df["relative_timeBd"] <= 24)]
first_24h_df["mrs_1y"] = first_24h_df["mrs_1y"].replace([0, 1, 2], "0-2")
first_24h_df["mrs_1y"] = first_24h_df["mrs_1y"].replace([3, 4, 5, 6], "3-6")

# dichotomize fisher score
first_24h_df["Fisher_Score"] = first_24h_df["Fisher_Score"].replace([1, 2], "1-2")
first_24h_df["Fisher_Score"] = first_24h_df["Fisher_Score"].replace([3, 4], "3-4")
# drop 0
first_24h_df = first_24h_df[first_24h_df["Fisher_Score"] != 0]

order = ["1-2", "3-4"]
sns.boxplot(x="Fisher_Score", y="systole", data=first_24h_df, hue="mrs_1y", ax=ax[0], legend=True, showfliers=False, order=order)
sns.boxplot(x="Fisher_Score", y="diastole", data=first_24h_df, hue="mrs_1y", ax=ax[1], legend=True, showfliers=False, order=order)
sns.boxplot(x="Fisher_Score", y="mitteldruck", data=first_24h_df, hue="mrs_1y", ax=ax[2], legend=True, showfliers=False, order=order)
plt.show()

# plot by fisher score and DCI
fig, ax = plt.subplots(3, 1, figsize=(10, 10))
first_24h_df = bp_df[(bp_df["relative_timeBd"] >= 0) & (bp_df["relative_timeBd"] <= 24)]
first_24h_df["mrs_1y"] = first_24h_df["mrs_1y"].replace([0, 1, 2], "0-2")
first_24h_df["mrs_1y"] = first_24h_df["mrs_1y"].replace([3, 4, 5, 6], "3-6")

# dichotomize fisher score
first_24h_df["Fisher_Score"] = first_24h_df["Fisher_Score"].replace([1, 2], "1-2")
first_24h_df["Fisher_Score"] = first_24h_df["Fisher_Score"].replace([3, 4], "3-4")
# drop 0
first_24h_df = first_24h_df[first_24h_df["Fisher_Score"] != 0]

order = ["1-2", "3-4"]
sns.boxplot(x="Fisher_Score", y="systole", data=first_24h_df, hue="DCI_ischemia", ax=ax[0], legend=True, showfliers=False, order=order)
sns.boxplot(x="Fisher_Score", y="diastole", data=first_24h_df, hue="DCI_ischemia", ax=ax[1], legend=True, showfliers=False, order=order)
sns.boxplot(x="Fisher_Score", y="mitteldruck", data=first_24h_df, hue="DCI_ischemia", ax=ax[2], legend=True, showfliers=False, order=order)
plt.show()

In [ ]:
registry_df[registry_df["DCI_ischemia"] == 1]["Fisher_Score"].value_counts()

In [ ]:
registry_df[registry_df["DCI_ischemia"] == 1]["WFNS"].value_counts()

In [ ]:
# - plot by WFNS score and DCI
fig, ax = plt.subplots(3, 1, figsize=(10, 10))
first_24h_df = bp_df[(bp_df["relative_timeBd"] >= 0) & (bp_df["relative_timeBd"] <= 24)]
first_24h_df["mrs_1y"] = first_24h_df["mrs_1y"].replace([0, 1, 2], "0-2")
first_24h_df["mrs_1y"] = first_24h_df["mrs_1y"].replace([3, 4, 5, 6], "3-6")

sns.boxplot(x="WFNS", y="systole", data=first_24h_df, hue="mrs_1y", ax=ax[0], legend=True, showfliers=False)
sns.boxplot(x="WFNS", y="diastole", data=first_24h_df, hue="mrs_1y", ax=ax[1], legend=True, showfliers=False)
sns.boxplot(x="WFNS", y="mitteldruck", data=first_24h_df, hue="mrs_1y", ax=ax[2], legend=True, showfliers=False)
plt.show()

# - plot by WFNS score and DCI
fig, ax = plt.subplots(3, 1, figsize=(10, 10))
sns.boxplot(x="WFNS", y="systole", data=first_24h_df, hue="DCI_ischemia", ax=ax[0], legend=True, showfliers=False)
sns.boxplot(x="WFNS", y="diastole", data=first_24h_df, hue="DCI_ischemia", ax=ax[1], legend=True, showfliers=False) 
sns.boxplot(x="WFNS", y="mitteldruck", data=first_24h_df, hue="DCI_ischemia", ax=ax[2], legend=True, showfliers=False)
plt.show()

# dichotomize WFNS into 1-2 and 3-5
first_24h_df["WFNS"] = first_24h_df["WFNS"].replace([1, 2], "1-2")
first_24h_df["WFNS"] = first_24h_df["WFNS"].replace([3, 4, 5], "3-5")

# plot by WFNS score and mrs
fig, ax = plt.subplots(3, 1, figsize=(10, 10))
sns.boxplot(x="WFNS", y="systole", data=first_24h_df, hue="mrs_1y", ax=ax[0], legend=True, showfliers=False)
sns.boxplot(x="WFNS", y="diastole", data=first_24h_df, hue="mrs_1y", ax=ax[1], legend=True, showfliers=False)
sns.boxplot(x="WFNS", y="mitteldruck", data=first_24h_df, hue="mrs_1y", ax=ax[2], legend=True, showfliers=False)
plt.show()
# plot by WFNS score and DCI
fig, ax = plt.subplots(3, 1, figsize=(10, 10))
sns.boxplot(x="WFNS", y="systole", data=first_24h_df, hue="DCI_ischemia", ax=ax[0], legend=True, showfliers=False)
sns.boxplot(x="WFNS", y="diastole", data=first_24h_df, hue="DCI_ischemia", ax=ax[1], legend=True, showfliers=False)
sns.boxplot(x="WFNS", y="mitteldruck", data=first_24h_df, hue="DCI_ischemia", ax=ax[2], legend=True, showfliers=False)
plt.show()

### Association BP with outcome 

Test association of BP in first 24h and DCI


In [ ]:
first_24h_df = bp_df[(bp_df["relative_timeBd"] >= 0) & (bp_df["relative_timeBd"] <= 24)]
# drop duplicates
first_24h_df = first_24h_df.drop_duplicates(subset=["pNr", "relative_timeBd"])
# dropna for systole, diastole, mitteldruck, DCI_ischemia
first_24h_df = first_24h_df.dropna(subset=["systole", "diastole", "mitteldruck", "DCI_ischemia"])

model = Lmer("DCI_ischemia ~ systole + (1|pNr)", data=first_24h_df, family="binomial")
model.fit()
print(model.summary())



In [ ]:
# pvalue for systole, diastole, mitteldruck
model.coefs

Test association of BP in first 24h and mrs

In [ ]:
import os
os.environ["R_HOME"] = "/Library/Frameworks/R.framework/Versions/4.1/Resources"
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
pandas2ri.activate()

first_24h_df = bp_df[(bp_df["relative_timeBd"] >= 0) & (bp_df["relative_timeBd"] <= 24)]
# drop duplicates
first_24h_df = first_24h_df.drop_duplicates(subset=["pNr", "relative_timeBd"])
# dropna for systole, diastole, mitteldruck, DCI_ischemia
first_24h_df = first_24h_df.dropna(subset=["systole", "diastole", "mitteldruck", "mrs_1y"])
# group per 1h
# first_24h_df["relative_timeBd_cat"] = first_24h_df["relative_timeBd"].apply(np.floor)
# # group by pnr and relative_timeBd_cat and get median of systole, diastole, mitteldruck
# first_24h_df = first_24h_df.groupby(["pNr", "relative_timeBd_cat"]).agg({"systole": "median", "diastole": "median", "mitteldruck": "median", "mrs_1y": "first"}).reset_index()

ro.globalenv['df'] = first_24h_df
ro.r('''
# Select CRAN mirror explicitly
options(repos = c(CRAN = "https://cran.r-project.org"))

# Install ordinal package if not installed
if (!require(ordinal)) {
    install.packages("ordinal")
}
library(ordinal)

df$response <- factor(df$mrs_1y, ordered=TRUE)
model <- clmm(response ~ mitteldruck + (1 | pNr), data=df)
summary(model)
''')

In [ ]:
first_24h_df = bp_df[(bp_df["relative_timeBd"] >= 0) & (bp_df["relative_timeBd"] <= 24)]
# drop duplicates
first_24h_df = first_24h_df.drop_duplicates(subset=["pNr", "relative_timeBd"])
# dropna for systole, diastole, mitteldruck, DCI_ischemia
first_24h_df = first_24h_df.dropna(subset=["systole", "diastole", "mitteldruck", "mrs_1y"])

# # group by pnr and get median of systole, diastole, mitteldruck
first_24h_df_median = first_24h_df.groupby(["pNr"]).agg({"systole": "median", "diastole": "median", "mitteldruck": "median", "mrs_1y": "first"}).reset_index()

from statsmodels.miscmodels.ordinal_model import OrderedModel

# fit model
model = OrderedModel.from_formula("mrs_1y ~ systole + diastole + mitteldruck", data=first_24h_df_median, distr="logit")
results = model.fit(method="bfgs", maxiter=1000)
print(results.summary())
print(results.pvalues)

use dichotomized mRS

In [ ]:
first_24h_df = bp_df[(bp_df["relative_timeBd"] >= 0) & (bp_df["relative_timeBd"] <= 24)]
# drop duplicates
first_24h_df = first_24h_df.drop_duplicates(subset=["pNr", "relative_timeBd"])
# dropna for systole, diastole, mitteldruck, DCI_ischemia
first_24h_df = first_24h_df.dropna(subset=["systole", "diastole", "mitteldruck", "mrs_1y"])
# dichotomize mrs_1y into 0-2 and 3-6
first_24h_df["mrs_1y_02"] = first_24h_df["mrs_1y"].isin([0, 1, 2]).astype(int)

# group per 1h
first_24h_df["relative_timeBd_cat"] = first_24h_df["relative_timeBd"].apply(np.floor)
# group by pnr and relative_timeBd_cat and get median of systole, diastole, mitteldruck
first_24h_df = first_24h_df.groupby(["pNr", "relative_timeBd_cat"]).agg({"systole": "median", "diastole": "median", "mitteldruck": "median", "mrs_1y_02": "first"}).reset_index()

model = Lmer("mrs_1y_02 ~ systole + diastole + mitteldruck + (1|pNr)", data=first_24h_df, family="binomial")
model.fit()
print(model.summary())

### BP per outcome category

In [ ]:
first_24h_df = bp_df[(bp_df["relative_timeBd"] >= 0) & (bp_df["relative_timeBd"] <= 24)]
# drop duplicates
first_24h_df = first_24h_df.drop_duplicates(subset=["pNr", "relative_timeBd"])
# dropna for systole, diastole, mitteldruck, DCI_ischemia
first_24h_df = first_24h_df.dropna(subset=["systole", "diastole", "mitteldruck", "mrs_1y"])
# dichotomize mrs_1y into 0-2 and 3-6
first_24h_df["mrs_1y_02"] = first_24h_df["mrs_1y"].isin([0, 1, 2]).astype(int)

sys_model = Lmer("systole ~ mrs_1y_02 + (1|pNr)", data=first_24h_df, family="gaussian")
sys_model.fit()
print(sys_model.summary())

dia_model = Lmer("diastole ~ mrs_1y_02 + (1|pNr)", data=first_24h_df, family="gaussian")
dia_model.fit()
print(dia_model.summary())

mean_model = Lmer("mitteldruck ~ mrs_1y_02 + (1|pNr)", data=first_24h_df, family="gaussian")
mean_model.fit()
print(mean_model.summary())



In [ ]:
first_24h_df = bp_df[(bp_df["relative_timeBd"] >= 0) & (bp_df["relative_timeBd"] <= 24)]
# drop duplicates
first_24h_df = first_24h_df.drop_duplicates(subset=["pNr", "relative_timeBd"])
# dropna for systole, diastole, mitteldruck, DCI_ischemia
first_24h_df = first_24h_df.dropna(subset=["systole", "diastole", "mitteldruck", "DCI_ischemia"])

sys_model = Lmer("systole ~ DCI_ischemia + (1|pNr)", data=first_24h_df, family="gaussian")
sys_model.fit()
print(sys_model.summary())

dia_model = Lmer("diastole ~ DCI_ischemia + (1|pNr)", data=first_24h_df, family="gaussian")
dia_model.fit()
print(dia_model.summary())

mean_model = Lmer("mitteldruck ~ DCI_ischemia + (1|pNr)", data=first_24h_df, family="gaussian")
mean_model.fit()
print(mean_model.summary())


# Summary

There is a significant difference in systolic BP in mrs 0-2 vs 3-6

No significant effect of BP on mrs
